In [0]:
from mlflow.deployments import get_deploy_client

In [0]:
client = get_deploy_client("databricks")

In [0]:
def get_or_create_endpoint(endpoint_name, config):
    try:
        # Try to get the existing endpoint
        existing = client.get_endpoint(endpoint_name)
        print(f"Endpoint '{endpoint_name}' already exists — updating...")
        return client.update_endpoint(
            endpoint=endpoint_name,
            config=config
        )
    except Exception as e:
        if "does not exist" in str(e).lower() or "not found" in str(e).lower():
            print(f"Endpoint '{endpoint_name}' does not exist — creating...")
            return client.create_endpoint(
                name=endpoint_name,
                config=config
            )
        else:
            raise e  # re-raise if it's a different error

In [0]:
# ── Config ─────────────────────────────────────────────────────────────────────
config = {
    "served_entities": [
        {
            "name": "hr-policy-agent",
            "entity_name": "workspace.hr.hr_policy_agent",  # full UC path
            "entity_version": "8",
            "workload_size": "Small",
            "scale_to_zero_enabled": True,
            "workload_type": "CPU",
            "environment_vars": {
                "GMAIL_SENDER":     "{{secrets/hr-agent-secrets/gmail-sender}}",
                "GMAIL_PASSWORD":   "{{secrets/hr-agent-secrets/gmail-app-password}}",
                "GMAIL_RECIPIENT":  "{{secrets/hr-agent-secrets/gmail-recipient}}"
            }
        }
    ],
    "traffic_config": {
        "routes": [
            {
                "served_model_name": "hr-policy-agent",
                "traffic_percentage": 100
            }
        ]
    }
}

# ── Run ────────────────────────────────────────────────────────────────────────
endpoint = get_or_create_endpoint("hr-policy-agent-endpoint", config)
print(endpoint)